# Velocity-Field-Based Double Image

This example shows how to create image B by applying a spatially varying displacement field to the particles from image A. The same workflow works for synthetic fields and real velocity data from CFD/PIV.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import RegularGridInterpolator

import synpivimage as spi

np.random.seed(42)

## 1) Camera, laser, and initial particles

In [ ]:
cam = spi.Camera(
    nx=512,
    ny=512,
    bit_depth=16,
    qe=1,
    sensitivity=1,
    baseline_noise=50,
    dark_noise=10,
    shot_noise=False,
    fill_ratio_x=1.0,
    fill_ratio_y=1.0,
    particle_image_diameter=2,
)

laser = spi.Laser(width=0.25, shape_factor=2)

particles = spi.Particles.generate(
    ppp=0.08,
    dx_max=[0.0, 6.0],
    dy_max=[-1.0, 1.0],
    dz_max=[0.0, 0.0],
    size=2.0,
    camera=cam,
    laser=laser,
)

In [ ]:
imgA, partA = spi.take_image(
    laser,
    cam,
    particles,
    particle_peak_count=1000,
)

## 2) Build a synthetic coarse velocity/displacement field

In this section, `u_field` and `v_field` are displacements in **pixel units for one frame pair**.

In [ ]:
x_field = np.linspace(0, cam.nx - 1, 16)
y_field = np.linspace(0, cam.ny - 1, 12)
xx_field, yy_field = np.meshgrid(x_field, y_field, indexing="xy")

u_field = 6.0 * (1.0 - ((yy_field - cam.ny / 2) / (cam.ny / 2)) ** 2)
v_field = 0.4 * np.sin(2 * np.pi * xx_field / cam.nx)

In [ ]:
plt.figure(figsize=(6, 5))
plt.quiver(xx_field, yy_field, u_field, v_field)
plt.gca().invert_yaxis()
plt.title("Coarse displacement field [px]")
plt.xlabel("x [px]")
plt.ylabel("y [px]")
plt.show()

## 3) Interpolate onto particle positions and generate image B

`RegularGridInterpolator` expects points in `(y, x)` order for arrays shaped as `(ny, nx)`.

In [ ]:
u_interp = RegularGridInterpolator(
    (y_field, x_field),
    u_field,
    method="linear",
    bounds_error=False,
    fill_value=0.0,
)
v_interp = RegularGridInterpolator(
    (y_field, x_field),
    v_field,
    method="linear",
    bounds_error=False,
    fill_value=0.0,
)

sample_points = np.column_stack((partA.y, partA.x))
dx = u_interp(sample_points)
dy = v_interp(sample_points)

In [ ]:
displaced_particles = partA.displace(dx=dx, dy=dy, dz=0.0)
imgB, partB = spi.take_image(
    laser,
    cam,
    displaced_particles,
    particle_peak_count=1000,
)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
axs[0].imshow(imgA, cmap="gray")
axs[0].set_title("Image A")
axs[1].imshow(imgB, cmap="gray")
axs[1].set_title("Image B (velocity-based shift)")
for ax in axs:
    ax.set_xlabel("x [px]")
axs[0].set_ylabel("y [px]")
plt.tight_layout()
plt.show()

In [ ]:
print(f"mean dx = {np.mean(dx):.3f} px, std dx = {np.std(dx):.3f} px")
print(f"mean dy = {np.mean(dy):.3f} px, std dy = {np.std(dy):.3f} px")

## 4) Use real or simulated velocity fields (m/s)

If your field is in physical units, convert to pixels before displacement:

- `u_px = u_mps * dt / meters_per_pixel`
- `v_px = v_mps * dt / meters_per_pixel`

Then interpolate `u_px`, `v_px` at particle coordinates exactly as above.

In [ ]:
dt = 20e-6  # s between frames
meters_per_pixel = 12e-6

x_real_px = np.linspace(0, cam.nx - 1, 32)
y_real_px = np.linspace(0, cam.ny - 1, 24)
xx_real, yy_real = np.meshgrid(x_real_px, y_real_px, indexing="xy")

u_real_mps = 0.7 * (1.0 - ((yy_real - cam.ny / 2) / (cam.ny / 2)) ** 2)
v_real_mps = 0.05 * np.sin(2 * np.pi * xx_real / cam.nx)

u_real_px = u_real_mps * dt / meters_per_pixel
v_real_px = v_real_mps * dt / meters_per_pixel

u_real_interp = RegularGridInterpolator((y_real_px, x_real_px), u_real_px, bounds_error=False, fill_value=0.0)
v_real_interp = RegularGridInterpolator((y_real_px, x_real_px), v_real_px, bounds_error=False, fill_value=0.0)

dx_real = u_real_interp(sample_points)
dy_real = v_real_interp(sample_points)

imgB_real, _ = spi.take_image(
    laser,
    cam,
    partA.displace(dx=dx_real, dy=dy_real, dz=0.0),
    particle_peak_count=1000,
)

print(f"real-field displacement range: dx=[{dx_real.min():.3f}, {dx_real.max():.3f}] px")